# OpenAlex Grant Publication Miner

Queries [OpenAlex](https://openalex.org/) for all publications acknowledging a given grant award.  
Exports results to CSV with title, DOI, journal, date, and BibTeX.

**Usage:** Change `AWARD_ID` and `FUNDER_OPENALEX_ID` in the config cell, then Run All.

In [ ]:
# nbconvert runs with cwd = src/; data files live in the repo root
import os
if os.path.basename(os.getcwd()) == "src":
    os.chdir("..")

In [1]:
import requests
import csv
import time
from datetime import datetime

try:
    from scholarly import scholarly as _gs
except ImportError:
    import subprocess
    subprocess.run(["uv", "pip", "install", "scholarly"], check=True)
    from scholarly import scholarly as _gs

try:
    import openpyxl
except ImportError:
    import subprocess
    subprocess.run(["uv", "pip", "install", "openpyxl"], check=True)
    import openpyxl

## Configuration

Common funder IDs (OpenAlex entity IDs — find others at [OpenAlex Funders](https://api.openalex.org/funders?search=)):
- **NSF**: `F4320306076`
- **DOE**: `F4320321988`
- **NIH**: `F4320332082`

In [2]:
# ── Config ──────────────────────────────────────────────
AWARD_ID = "2309043"                    # just the numeric part
FUNDER_ID = "F4320306076"               # NSF (OpenAlex entity ID)
EMAIL = "ghollyer@seas.upenn.edu"          # polite pool — faster rate limit
OUTPUT_CSV = "grant_publications.csv"
# ────────────────────────────────────────────────────────

In [3]:
def fetch_all_works(funder_id: str, award_id: str, email: str) -> list[dict]:
    base_url = "https://api.openalex.org/works"

    params = {
        "filter": f"awards.funder_id:{funder_id},awards.funder_award_id:{award_id}",
        "per_page": 200,
        "cursor": "*",
        "mailto": email,
    }

    all_works = []
    page = 0

    while True:
        r = requests.get(base_url, params=params)
        r.raise_for_status()
        data = r.json()

        results = data.get("results", [])
        all_works.extend(results)
        page += 1

        total = data.get("meta", {}).get("count", "?")
        print(f"  Page {page}: fetched {len(results)} works (total so far: {len(all_works)}/{total})")

        next_cursor = data.get("meta", {}).get("next_cursor")
        if not next_cursor or len(results) == 0:
            break

        params["cursor"] = next_cursor
        time.sleep(0.15)

    return all_works

In [4]:
def extract_authors(work: dict) -> list[str]:
    """Pull author display names in order."""
    return [
        a.get("author", {}).get("display_name", "Unknown")
        for a in work.get("authorships", [])
    ]


def make_bibtex(work: dict) -> str:
    """
    Generate a minimal BibTeX entry from an OpenAlex work object.
    
    Uses the OpenAlex ID suffix as the cite key since it's unique.
    Journal articles get @article, everything else gets @misc.
    """
    oa_id = work.get("id", "").split("/")[-1]  # e.g. W1234567890
    authors = extract_authors(work)
    title = work.get("title", "")
    doi = work.get("doi", "") or ""
    year = work.get("publication_year", "")
    
    # Journal info lives in primary_location.source
    source = (work.get("primary_location") or {}).get("source") or {}
    journal = source.get("display_name", "")
    
    volume = work.get("biblio", {}).get("volume", "") or ""
    issue = work.get("biblio", {}).get("issue", "") or ""
    first_page = work.get("biblio", {}).get("first_page", "") or ""
    last_page = work.get("biblio", {}).get("last_page", "") or ""
    pages = f"{first_page}--{last_page}" if first_page and last_page else first_page
    
    entry_type = "article" if work.get("type") == "journal-article" else "misc"
    
    # Build field list, skip empties
    fields = []
    if authors:  fields.append(f'  author = {{{" and ".join(authors)}}}')
    if title:    fields.append(f'  title = {{{title}}}')
    if journal:  fields.append(f'  journal = {{{journal}}}')
    if year:     fields.append(f'  year = {{{year}}}')
    if volume:   fields.append(f'  volume = {{{volume}}}')
    if issue:    fields.append(f'  number = {{{issue}}}')
    if pages:    fields.append(f'  pages = {{{pages}}}')
    if doi:      fields.append(f'  doi = {{{doi.replace("https://doi.org/", "")}}}')
    
    return f"@{entry_type}{{{oa_id},\n" + ",\n".join(fields) + "\n}"


def parse_work(work: dict) -> dict:
    """Flatten an OpenAlex work into a row dict for CSV export."""
    source = (work.get("primary_location") or {}).get("source") or {}
    return {
        "title": work.get("title", ""),
        "doi": work.get("doi", "") or "",
        "journal": source.get("display_name", ""),
        "date": work.get("publication_date", ""),
        "year": work.get("publication_year", ""),
        "authors": "; ".join(extract_authors(work)),
        "type": work.get("type", ""),
        "cited_by_count": work.get("cited_by_count", 0),
        "open_access": work.get("open_access", {}).get("is_oa", False),
        "source": "OpenAlex",
        "bibtex": make_bibtex(work),
    }


def _scholar_query(query: str, max_results: int) -> list[dict]:
    """Run a single Google Scholar query and return raw result dicts."""
    results = []
    try:
        for i, pub in enumerate(_gs.search_pubs(f'"{query}"')):
            if i >= max_results:
                break
            bib = pub.get("bib", {})
            title = bib.get("title", "")
            raw_authors = bib.get("author", "")
            authors = "; ".join(raw_authors) if isinstance(raw_authors, list) else (str(raw_authors) if raw_authors else "")
            year = str(bib.get("pub_year", ""))
            journal = bib.get("venue", "") or bib.get("journal", "") or ""
            url = pub.get("pub_url", "") or ""
            doi = ("https://doi.org/" + url.split("doi.org/")[-1]) if "doi.org/" in url else ""
            results.append({
                "title": title,
                "doi": doi,
                "journal": journal,
                "date": f"{year}-01-01" if year else "",
                "year": int(year) if year.isdigit() else "",
                "authors": authors,
                "type": "journal-article",
                "cited_by_count": pub.get("num_citations", 0),
                "open_access": False,
                "source": "Google Scholar only",
                "bibtex": "",
            })
            time.sleep(0.5)
    except Exception as e:
        print(f"  Scholar error for '{query}': {e}")
    return results


def fetch_scholar_works(award_id: str, max_results: int = 40) -> list[dict]:
    """
    Search Google Scholar across multiple grant-number variants and return
    deduplicated results.

    Variants searched (in addition to the bare award_id):
      DMR-<award_id>   e.g. DMR-2309043  (standard NSF format)
      DMR<award_id>    e.g. DMR2309043   (no hyphen)
      <yy>-<rest>      e.g. 23-09043     (year-hyphenated)
      DMR<yy>-<rest>   e.g. DMR23-09043  (prefix + year-hyphenated)
    """
    # Build variant list
    queries = [award_id]
    queries.append(f"DMR-{award_id}")
    queries.append(f"DMR{award_id}")
    if len(award_id) >= 2:
        yy, rest = award_id[:2], award_id[2:]
        queries.append(f"{yy}-{rest}")
        queries.append(f"DMR{yy}-{rest}")

    seen_titles: set[str] = set()
    all_results: list[dict] = []

    for q in queries:
        print(f"  Searching Scholar: \"{q}\"")
        hits = _scholar_query(q, max_results)
        new = 0
        for pub in hits:
            norm = pub["title"].lower().strip()
            if norm and norm not in seen_titles:
                seen_titles.add(norm)
                all_results.append(pub)
                new += 1
                print(f"    + {pub['title'][:70]}")
        print(f"    → {new} new result(s)")

    return all_results


def merge_and_dedup(oa_rows: list[dict], scholar_pubs: list[dict]) -> list[dict]:
    """Merge OpenAlex and Scholar results, deduplicating by DOI then title."""
    def norm_doi(doi: str) -> str:
        return doi.lower().replace("https://doi.org/", "").strip()

    oa_dois = {norm_doi(r["doi"]) for r in oa_rows if r["doi"]}
    oa_titles = {r["title"].lower().strip() for r in oa_rows if r["title"]}

    merged = list(oa_rows)
    for pub in scholar_pubs:
        doi = norm_doi(pub["doi"])
        title = pub["title"].lower().strip()
        if doi and doi in oa_dois:
            continue
        if title and title in oa_titles:
            continue
        merged.append(pub)

    return merged

In [5]:
# ── Fetch: OpenAlex ─────────────────────────────────────
print(f"Querying OpenAlex for funder={FUNDER_ID}, award={AWARD_ID}...\n")
works = fetch_all_works(FUNDER_ID, AWARD_ID, EMAIL)
oa_rows = [parse_work(w) for w in works]
print(f"OpenAlex total: {len(oa_rows)}\n")

# ── Fetch: Google Scholar ────────────────────────────────
print(f"Searching Google Scholar for \"{AWARD_ID}\"...\n")
scholar_pubs = fetch_scholar_works(AWARD_ID)
print(f"\nScholar candidates: {len(scholar_pubs)}\n")

# ── Merge & deduplicate ──────────────────────────────────
rows = merge_and_dedup(oa_rows, scholar_pubs)
new_count = len(rows) - len(oa_rows)
print(f"Total after merge: {len(rows)}  ({new_count} new from Google Scholar only)")

Querying OpenAlex for funder=F4320306076, award=2309043...



  Page 1: fetched 75 works (total so far: 75/75)
OpenAlex total: 75

Searching Google Scholar for "2309043"...

  Searching Scholar: "2309043"


    + Materials Research Science and Engineering Centers (MRSEC) UPENN
    + Classification of middle-and long-distance runners based upon their pe
    + Tuning Breakdown-to-Coercive Field Ratio in Ultra-Thin Al1-xScxN Films
    + An electrodiffusive network model with multicompartmental neurons and 
    + Controlling Assembly of Polymer-Grafted Nanoparticles to Enhance Mecha
    + Diffusion of Bottlebrush Polymer-Grafted Nanoparticles: The Role of Ma
    + Rheology and Clogging Study of Filamentous Suspensions: Bridging Micro
    + Bioinspired switchable passive daytime radiative cooling coatings
    + Amphiphilic nanopores that condense undersaturated water vapor and exu
    + 3D Printing of Bicontinuous Nanoparticle‐Stabilized Emulsion Gels via 
    + Spontaneous assembly of condensate networks during the demixing of str
    + Reversible nanopore sealing and in situ iron oxide nanoparticle synthe
    + Nanoparticle Diffusion in Crowded Polymer Nanocomposite Melts
    + Tissue-like c

    + 53BP1 condensates function as bioreactors for NHEJ directed DNA repair
    + Building rigid networks with prestress and selective pruning
    + Broadband light harvesting from scalable two-dimensional semiconductor
    + Contrastive power-efficient physical learning in resistor networks
    + Ionic Conductivity in Solvent-Swollen Surfactant-Like Multiblock Copol
    + Flexible antireflection coatings with enhanced durability and antifogg
    + Effect of localized control of cross-link density on mechanical proper
    + The Polymer Structure–Ion Conductivity–Mechanical Property Relationshi
    + Decoupling CO2 effects from electrochemistry: A mechanistic study of c
    + Stochastic dynamics of granular hopper flows: A configurational mode c
    + Determination of film thicknesses of metal oxides prepared by atomic l
    → 11 new result(s)
  Searching Scholar: "DMR2309043"


    + Low-Field Ferroelectricity in 10 nm AlBScN Thin Films
    + Structural complexity driven by liquid–liquid crystal phase separation
    + Reversible Physical Gelation of Thermotropic Liquid Crystals Driven by
    + Self-construction and destruction of living transport networks
    + Photoelectrocatalytic reduction of CO2 to formate using immobilized mo
    + Defect dynamics of colloidal antiferromagnetic tetratic phase confined
    + Coalescence of Sessile Radial Nematic Droplets
    → 7 new result(s)
  Searching Scholar: "23-09043"


    + Genome-Wide Identification and Analysis of the Growth-Regulating Facto
    + Early fire detection using long short-term memory-based instance segme
    + PD-0970 Biodistribution Data of ImmunoPET: A Phase 0/1 Study Character
    → 3 new result(s)
  Searching Scholar: "DMR23-09043"


    → 0 new result(s)

Scholar candidates: 61

Total after merge: 123  (48 new from Google Scholar only)


In [6]:
# ── Display ─────────────────────────────────────────────
rows.sort(key=lambda x: x["date"], reverse=True)

for i, r in enumerate(rows, 1):
    oa_flag = " [OA]" if r["open_access"] else ""
    src = r.get("source", "OpenAlex")
    src_flag = " [Scholar only]" if src == "Google Scholar only" else ""
    print(f"{i}. ({r['year']}) {r['title'][:80]}")
    print(f"   {r['journal']} | cited: {r['cited_by_count']}{oa_flag}{src_flag}")
    print(f"   {r['doi']}\n")

1. (2026) Tungsten doping-induced phase transition in CVD-grown MoS2 bilayers
   npj 2D Materials and Applications | cited: 0 [OA]
   https://doi.org/10.1038/s41699-026-00707-z

2. (2026) Magnetically Actuated Drops‐on‐Stilts Microrobots for Transport and Delivery of 
   Advanced Materials Interfaces | cited: 0 [OA]
   https://doi.org/10.1002/admi.70508

3. (2026) Anion transport and selectivity in ordered nanoporous polymers with 1 nm scale c
   ChemRxiv | cited: 0 [OA]
   https://doi.org/10.26434/chemrxiv.15001556/v1

4. (2026) Programmable Non-Covalent Assembly of Disordered Peptides to Build Condensed Mat
   ChemRxiv | cited: 0 [OA]
   https://doi.org/10.26434/chemrxiv.15001095/v1

5. (2026) Site-selective enhancement of Eu emission in delta-doped GaN
   Applied Physics Letters | cited: 0 [OA]
   https://doi.org/10.1063/5.0317065

6. (2026) Microfluidic nanomagnetically isolated neuron- and astrocyte-derived extracellul
   npj Biosensing | cited: 0 [OA]
   https://doi.org/10.1038/s

In [7]:
# ── Export CSV ──────────────────────────────────────────
fieldnames = ["title", "doi", "journal", "date", "year", "authors",
              "type", "cited_by_count", "open_access", "source", "bibtex"]

with open(OUTPUT_CSV, "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(rows)

print(f"Wrote {len(rows)} records to {OUTPUT_CSV}")

try:
    from google.colab import files
    files.download(OUTPUT_CSV)
except ImportError:
    print(f"Not in Colab — file saved locally as {OUTPUT_CSV}")

Wrote 123 records to grant_publications.csv
Not in Colab — file saved locally as grant_publications.csv


## Caveats

- **Coverage**: OpenAlex relies on Crossref Fundref metadata. If a publisher didn't deposit structured grant info, the paper won't appear here even if the acknowledgment section mentions the grant. Supplement with a Google Scholar search for `"DMR-2309043"` in quotes.
- **Award ID matching**: Some papers report the full program prefix (e.g., `DMR-2309043`) while OpenAlex may strip it to just `2309043`. The API handles this, but edge cases exist.
- **BibTeX quality**: Generated from OpenAlex metadata, which is generally good but occasionally has encoding artifacts or missing fields. Spot-check before submitting to a journal.

## Web of Science Cross-Reference

Compares `grant_publications.csv` against the WoS export (`savedrecs (2).xlsx`).  
Matches first by DOI, then by normalised title.  
Produces a summary and writes `crossref_report.csv` with `in_openalex` / `in_wos` flags and WoS citation counts.

In [ ]:
import csv
import openpyxl

WOS_FILE    = "savedrecs_05_29.xlsx"
OA_CSV      = OUTPUT_CSV          # grant_publications.csv from the cell above
REPORT_CSV  = "crossref_report.csv"

# ── Helpers ──────────────────────────────────────────────────────────────────

def norm_doi(doi: str) -> str:
    return (doi or "").lower().replace("https://doi.org/", "").replace("http://doi.org/", "").strip()

def norm_title(title: str) -> str:
    import re
    return re.sub(r"\s+", " ", (title or "").lower().strip())


# ── Load WoS ─────────────────────────────────────────────────────────────────
# New export format: row 1 = headers, data from row 2

wb  = openpyxl.load_workbook(WOS_FILE, read_only=True, data_only=True)
ws  = wb.active

wos_headers = None
wos_rows    = []

for i, row in enumerate(ws.iter_rows(values_only=True), start=1):
    if i == 1:
        wos_headers = list(row)
    elif wos_headers:
        rec = dict(zip(wos_headers, row))
        if rec.get("Article Title"):
            wos_rows.append(rec)

wb.close()
print(f"WoS records loaded: {len(wos_rows)}")

# Build lookup indexes for WoS
wos_by_doi   = {}
wos_by_title = {}

for rec in wos_rows:
    d = norm_doi(rec.get("DOI", "") or "")
    t = norm_title(rec.get("Article Title", "") or "")
    if d:
        wos_by_doi[d] = rec
    if t:
        wos_by_title[t] = rec


# ── Load OpenAlex/Scholar CSV ─────────────────────────────────────────────────

with open(OA_CSV, newline="", encoding="utf-8") as f:
    oa_rows = list(csv.DictReader(f))

print(f"OpenAlex+Scholar records loaded: {len(oa_rows)}")


# ── Cross-reference ───────────────────────────────────────────────────────────

def find_wos_match(row: dict) -> dict | None:
    d = norm_doi(row.get("doi", ""))
    t = norm_title(row.get("title", ""))
    if d and d in wos_by_doi:
        return wos_by_doi[d]
    if t and t in wos_by_title:
        return wos_by_title[t]
    return None

report_rows = []
wos_matched_titles = set()

for row in oa_rows:
    wos = find_wos_match(row)
    entry = {
        "title":          row.get("title", ""),
        "doi":            row.get("doi", ""),
        "journal":        row.get("journal", ""),
        "year":           row.get("year", ""),
        "authors":        row.get("authors", ""),
        "in_openalex":    True,
        "in_wos":         wos is not None,
        "wos_citations":  wos.get("Times Cited, WoS Core", "") if wos else "",
        "oa_citations":   row.get("cited_by_count", ""),
        "open_access":    row.get("open_access", ""),
        "source":         row.get("source", ""),
    }
    report_rows.append(entry)
    if wos:
        wos_matched_titles.add(norm_title(wos.get("Article Title", "") or ""))

# WoS-only side
for rec in wos_rows:
    t = norm_title(rec.get("Article Title", "") or "")
    d = norm_doi(rec.get("DOI", "") or "")
    already_matched = (
        t in wos_matched_titles
        or (d and d in {norm_doi(r.get("doi","")) for r in oa_rows})
    )
    if not already_matched:
        report_rows.append({
            "title":         rec.get("Article Title", ""),
            "doi":           rec.get("DOI", "") or "",
            "journal":       rec.get("Source Title", ""),
            "year":          str(rec.get("Publication Year", "") or ""),
            "authors":       rec.get("Authors", ""),
            "in_openalex":   False,
            "in_wos":        True,
            "wos_citations": rec.get("Times Cited, WoS Core", ""),
            "oa_citations":  "",
            "open_access":   "",
            "source":        "WoS only",
        })

report_rows.sort(key=lambda r: (not (r["in_openalex"] and r["in_wos"]), str(r.get("year", "") or "")), reverse=False)


# ── Summary ───────────────────────────────────────────────────────────────────

in_both   = sum(1 for r in report_rows if r["in_openalex"] and r["in_wos"])
oa_only   = sum(1 for r in report_rows if r["in_openalex"] and not r["in_wos"])
wos_only  = sum(1 for r in report_rows if not r["in_openalex"] and r["in_wos"])

print(f"\n{'─'*50}")
print(f"  In both OpenAlex/Scholar AND WoS : {in_both}")
print(f"  OpenAlex/Scholar only            : {oa_only}")
print(f"  WoS only                         : {wos_only}")
print(f"  Total unique publications        : {len(report_rows)}")
print(f"{'─'*50}\n")

if oa_only:
    print("OpenAlex/Scholar only (not in WoS):")
    for r in report_rows:
        if r["in_openalex"] and not r["in_wos"]:
            print(f"  [{r['year']}] {r['title'][:80]}")

if wos_only:
    print("\nWoS only (not in OpenAlex/Scholar):")
    for r in report_rows:
        if not r["in_openalex"] and r["in_wos"]:
            print(f"  [{r['year']}] {r['title'][:80]}")


# ── Export ────────────────────────────────────────────────────────────────────

fieldnames = ["title", "doi", "journal", "year", "authors",
              "in_openalex", "in_wos", "wos_citations", "oa_citations",
              "open_access", "source"]

with open(REPORT_CSV, "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(report_rows)

print(f"\nWrote {len(report_rows)} records to {REPORT_CSV}")